# Stretch: more questions

For when the ladder of ten is done and there is time left. These get
harder as you go, so nobody is expected to finish them all.

Each one is a real question about the listening data. Before you run
anything, say out loud roughly what you expect the answer to be. That is
how you notice when a query that runs fine is answering the wrong
question, and two of these are built to catch you exactly like that.

Two of them need something new, and the notebook tells you what it is
when you get there. No joins yet: that is next session.

In [ ]:
import os
import sqlite3
from pathlib import Path

here = Path.cwd()
while not (here / "data" / "music.db").exists() and here != here.parent:
    here = here.parent
os.chdir(here)

con = sqlite3.connect("data/music.db")


def run(sql, limit=10):
    """Run a query and print the rows as an aligned table."""
    cursor = con.execute(sql)
    headers = [d[0] for d in cursor.description]
    rows = cursor.fetchall()
    widths = [max(len(str(h)), *(len(str(r[i])) for r in rows[:limit] or [[""]]))
              for i, h in enumerate(headers)]
    print("  ".join(str(h).ljust(w) for h, w in zip(headers, widths)))
    print("  ".join("-" * w for w in widths))
    for row in rows[:limit]:
        print("  ".join(str(v).ljust(w) for v, w in zip(row, widths)))
    print(f"({len(rows)} rows)" if len(rows) <= limit
          else f"... {len(rows)} rows in total")

## Stretch 1: the third table

There is a table nobody has looked at yet: `awards`. Have a look at it
first.

In [ ]:
run("SELECT * FROM awards LIMIT 5")

Two questions:

1. Which award has been handed out most often?
2. Which artists have won the most awards? And did any of them win the
   **same** award more than once?

For the second one you need two different counts per artist, side by
side. You have seen both of them already.

In [ ]:
run("""
""")

In [ ]:
run("""
""")

The `awards` table only has an `artist_id`. Can you find the names of
the artists you found, with a second query on `artists`?

In [ ]:
run("""
""")

## Stretch 2: how many artists are still active?

`still_active` is `1` for yes and `0` for no. Here are three ways to ask
the question. Before running them, predict: do they all give the same
answer?

```sql
SELECT COUNT(still_active) FROM artists
SELECT SUM(still_active) FROM artists
SELECT COUNT(*) FROM artists WHERE still_active = 1
```

In [ ]:
run("""
""")

In [ ]:
run("""
""")

In [ ]:
run("""
""")

Which ones are right, and **why** is the odd one out wrong?

Then count the artists who are **not** active, using
`WHERE still_active != 1`. Add the two answers together. Is that the
number of artists? If not, where did the missing one go?

In [ ]:
run("""
""")

*Your answer here.*

## Stretch 3: artists by decade

In which decade were most of the artists formed?

There is no decade column, so you have to make one. Think back to the
integer division warning: in SQLite `1996 / 10` is `199`, not `199.6`.
That is usually a nuisance. Here it is exactly what you want.

In [ ]:
run("""
""")

One row in your answer will look odd. What is it, and should it be
there?

## Stretch 4: tracks played on every device

Which tracks have been played on all five devices? Show the track, the
number of devices and the number of plays, most played first.

In [ ]:
run("""
""")

Now the same, but only counting plays that were **not skipped**. A
track that was only ever skipped in the car should not count as played
in the car.

Where does `skipped = 0` go: in the `WHERE` or in the `HAVING`? Try
both. Do they give the same number of rows? Which one is right, and why?

In [ ]:
run("""
""")

In [ ]:
run("""
""")

*Your answer here.*

## Stretch 5: do I listen more at the weekend?

Two new pieces for this one.

**`strftime('%w', played_at)`** gives the day of the week, as text:
`'0'` for Sunday, `'1'` for Monday, up to `'6'` for Saturday. (You may
have met `strftime('%Y', ...)` for the year in the homework. Same
function, different letter.)

**`CASE`** is SQL's `if`. It makes a new value for each row:

```sql
CASE
    WHEN minutes_played < 1 THEN 'short'
    ELSE 'long'
END AS length
```

Try both in the cell below first.

In [ ]:
run("""
SELECT played_at,
       strftime('%w', played_at) AS weekday,
       minutes_played,
       CASE
           WHEN minutes_played < 1 THEN 'short'
           ELSE 'long'
       END AS length
FROM plays
LIMIT 5
""")

Now write a query with one row for `'weekend'` and one for `'weekday'`,
and the number of plays in each. You can `GROUP BY` the name you gave
your `CASE`.

Then look at the answer and ask: **is that a fair comparison?** If not,
add a column that makes it fair. (`COUNT(DISTINCT played_at)` counts
days.)

In [ ]:
run("""
""")

*Your answer here.*

## Stretch 6: which device lost listeners?

In the homework you found 1,122 plays in 2024 and 1,061 in 2025. Which
device is responsible for the drop?

Make one row per device, with a column for 2024 and a column for 2025,
like this:

```
device   plays_2024  plays_2025
-------  ----------  ----------
phone    ...         ...
```

Hint: `SUM(skipped)` counts skipped plays because `skipped` is a 0 or a
1. A `CASE` can make a 0 or a 1 out of anything you like.

In [ ]:
run("""
""")

**Harder:** add a column with the change from 2024 to 2025, and sort by
it so the biggest drop comes first.

In [ ]:
run("""
""")

In [ ]:
con.close()